# 2_Training.ipynb — Image Captioning Training
# Filled to match image_captioning.ipynb configuration exactly.
# Architecture : EncoderCNN (ResNet-50 frozen) + DecoderRNN (Bahdanau attention LSTM)
# Hyperparameters match CFG in image_captioning.ipynb


In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import sys
sys.path.append('/opt/cocoapi/PythonAPI')
from pycocotools.coco import COCO
from data_loader import get_loader
from model import EncoderCNN, DecoderRNN
import math
import os
import numpy as np
import time


In [ ]:
# ── Hyperparameters (matched to image_captioning.ipynb CFG) ──────────────────

batch_size      = 64       # CFG['batch_size'] = 64
vocab_threshold = 5        # CFG['min_freq']   = 5  → same word-frequency cutoff
vocab_from_file = False    # Set True after vocab.pkl is built on first run
embed_size      = 256      # CFG['embed_dim']  = 256
hidden_size     = 512      # CFG['hidden_dim'] = 512
num_epochs      = 10       # CFG['epochs']     = 10
save_every      = 1        # save checkpoint every epoch
print_every     = 100      # print loss every N steps
log_file        = 'training_log.txt'

os.makedirs('./models', exist_ok=True)


In [ ]:
# ── Image transforms (identical to ImageCaptionDataset train split) ──────────

transform_train = transforms.Compose([
    transforms.Resize(256),                          # resize shorter edge to 256
    transforms.RandomCrop(224),                      # random 224×224 crop
    transforms.RandomHorizontalFlip(),               # 50 % horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), # ImageNet mean
                         std =(0.229, 0.224, 0.225)) # ImageNet std
])


In [ ]:
# ── Data loader ──────────────────────────────────────────────────────────────

data_loader = get_loader(
    transform       = transform_train,
    mode            = 'train',
    batch_size      = batch_size,
    vocab_threshold = vocab_threshold,
    vocab_from_file = vocab_from_file,
)

vocab_size = len(data_loader.dataset.vocab)
print(f'Vocabulary size: {vocab_size}')


In [ ]:
# ── Model initialisation ─────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size).to(device)


In [ ]:
# ── Loss, optimiser, trainable parameters ────────────────────────────────────

criterion = nn.CrossEntropyLoss()

# Trainable params:
#   • All decoder parameters (attention, LSTM, FC, embeddings)
#   • Encoder projection head only (backbone is frozen in EncoderCNN)
params = (
    list(decoder.parameters()) +
    list(encoder.embed.parameters()) +
    list(encoder.bn.parameters())
)
print(f'Trainable parameter tensors: {len(params)}')
print(f'Trainable parameters: {sum(p.numel() for p in params):,}')

optimizer = torch.optim.Adam(params, lr=3e-4)   # CFG['lr'] = 3e-4

# Optional: cosine LR schedule over epochs (uncomment to enable)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)


In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────

total_step = math.ceil(
    len(data_loader.dataset.caption_lengths) /
    data_loader.batch_sampler.batch_size
)
print(f'Total steps per epoch: {total_step}')

import torch.utils.data as data

f = open(log_file, 'w')

for epoch in range(1, num_epochs + 1):
    epoch_start = time.time()
    running_loss = 0.0

    for i_step in range(1, total_step + 1):

        # ── Sample a batch of same-length captions ──────────────────────
        indices    = data_loader.dataset.get_train_indices()
        new_sampler = data.sampler.SubsetRandomSampler(indices=indices)
        data_loader.batch_sampler.sampler = new_sampler

        images, captions = next(iter(data_loader))
        images   = images.to(device)
        captions = captions.to(device)

        # ── Forward pass ─────────────────────────────────────────────────
        encoder.zero_grad()
        decoder.zero_grad()

        features = encoder(images)                    # (B, embed_size)
        outputs  = decoder(features, captions)        # (B, T, vocab_size)

        # outputs[:,t] predicts captions[:,t] → shapes align for view(-1,…)
        loss = criterion(outputs.view(-1, vocab_size), captions.view(-1))

        # ── Backward pass ─────────────────────────────────────────────────
        loss.backward()
        # Gradient clipping (CFG['clip'] = 1.0)
        nn.utils.clip_grad_norm_(params, max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()

        stats = (
            f'Epoch [{epoch}/{num_epochs}], '
            f'Step [{i_step}/{total_step}], '
            f'Loss: {loss.item():.4f}, '
            f'Perplexity: {np.exp(loss.item()):5.4f}'
        )
        print('\r' + stats, end='')
        sys.stdout.flush()
        f.write(stats + '\n')
        f.flush()

        if i_step % print_every == 0:
            print('\r' + stats)

    epoch_loss = running_loss / total_step
    epoch_time = time.time() - epoch_start
    print(f'\nEpoch {epoch} complete — avg loss: {epoch_loss:.4f}  ({epoch_time:.1f}s)')

    # ── Save checkpoint ───────────────────────────────────────────────
    if epoch % save_every == 0:
        torch.save(decoder.state_dict(),
                   os.path.join('./models', f'decoder-{epoch}.pkl'))
        torch.save(encoder.state_dict(),
                   os.path.join('./models', f'encoder-{epoch}.pkl'))
        print(f'  → Saved decoder-{epoch}.pkl, encoder-{epoch}.pkl')

f.close()
print('Training complete.')
